In [10]:
import rasterio as rio
from rasterio.merge import merge


In [3]:
path_rsimg = 'data/data-section-6/s2_10m_chenggong.tif'
path_clip_1 = 'data/data-section-6/s2_10m_chenggong_clip1.tif'
path_clip_2 = 'data/data-section-6/s2_10m_chenggong_clip2.tif'


In [4]:
clip_bounds_1 = [278726, 2748552, 282667, 2752483]
clip_bounds_2 = [281400, 2747170, 284193, 2751081]


In [5]:
with rio.open(path_rsimg) as src:
    ## 计算裁剪范围的像素窗口
    window = src.window(*clip_bounds_2)
    window = window.round_offsets().round_shape()

    ## 读取窗口内数据
    data = src.read(window=window)

    ## 计算裁剪数据的转换矩阵
    transform_clip = src.window_transform(window)

    with rio.open(
        fp= path_clip_2,
        mode='w',
        driver='GTiff',
        height= window.height,
        width= window.width,
        count=src.count,
        dtype=data.dtype,
        crs=src.crs,
        transform=transform_clip) as dst:
        ## 写入数据
        dst.write(data)


c:\Users\HP\miniconda3\envs\myenv\Lib\site-packages\rasterio\windows.py:729: RasterioDeprecationWarning: round_shape is deprecated and will be removed in Rasterio 2.0.0.
  warnings.warn(


In [17]:
path_mosaic = 'data/data-section-6/s2_10m_chenggong_mosaic.tif'


In [18]:
src_1 = rio.open(path_clip_1)
src_2 = rio.open(path_clip_2)
merged_data, merged_transform =  merge([src_1, src_2])
meta_mosaic = src_1.meta.copy()

meta_mosaic.update({
    'height': merged_data.shape[1],
    'width': merged_data.shape[2],
    'transform': merged_transform,
})
meta_mosaic



{'driver': 'GTiff',
 'dtype': 'uint16',
 'nodata': None,
 'width': 547,
 'height': 531,
 'count': 4,
 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 48N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",105],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]'),
 'transform': Affine(10.0, 0.0, 278720.0,
        0.0, -10.0, 2752490.0)}

In [19]:
with rio.open(
    fp=path_mosaic,
    mode='w',
    driver='GTiff',
    height=meta_mosaic['height'],
    width=meta_mosaic['width'],
    count=meta_mosaic['count'],
    dtype = meta_mosaic['dtype'],
    crs=meta_mosaic['crs'],
    transform=meta_mosaic['transform']
) as dst:
    dst.write(merged_data)


In [20]:
src_1.close()
src_2.close()

